# Postcards from my Jungle: sound generation

Sixteen imagined animal calls with **Stable Audio Open 1.0**, adapted from the course notebook (W05_03, first model). I chose this model over *Stable Audio 3 Small-Music* because Open 1.0 was trained on sound effects and field recordings, which suits animal calls, whereas the Small-Music model is aimed at music.

**Before running**
1. Runtime → GPU.
2. Accept the licence on the [model page](https://huggingface.co/stabilityai/stable-audio-open-1.0) (it is gated).
3. Add a Hugging Face read token as a Colab secret named `HF_TOKEN`, or paste it when asked.

In [ ]:
!pip uninstall -y torchao
!pip install -q torchsde -U diffusers

In [ ]:
import torch, soundfile as sf, numpy as np, os, shutil
from diffusers import StableAudioPipeline
from huggingface_hub import login
from IPython.display import Audio, display

login()   # skipped automatically if HF_TOKEN is set in Colab secrets

pipe = StableAudioPipeline.from_pretrained(
    "stabilityai/stable-audio-open-1.0", torch_dtype=torch.float16).to("cuda")
SR = pipe.vae.sampling_rate

## The 16 creatures

Each entry is `(name, prompt, duration in seconds)`. The names and order match the website (`script.js`), so `sound_00.wav` belongs to the first creature, and so on. Durations range from 2 to 20 seconds. Prompts describe a *real-sounding* source (frog, owl, bamboo) plus a setting, because the model responds better to concrete sound descriptions than to invented species.

In [ ]:
VOICES = [
 [
  "Mossgrumble",
  "deep resonant frog bellow in a wet rainforest, gurgling, low frequency, field recording",
  3
 ],
 [
  "Tinkerwing",
  "tiny high-pitched chittering of an insect-bird, fast fluttering wings, jungle canopy",
  5
 ],
 [
  "Pondwhistle",
  "bubbly whistling call of a pond creature, liquid pops and soft peeps, swamp ambience",
  4
 ],
 [
  "Dusk Howler",
  "long mournful howl of a large primate at sunset, echoing across a jungle valley",
  8
 ],
 [
  "Clickerjack",
  "rapid sharp clicking and clacking alarm call, wooden clicks, nervous creature, rainforest",
  12
 ],
 [
  "Velvet Purrfrog",
  "soft purring croak of a furry frog, warm rumbling, light rain in the background",
  6
 ],
 [
  "Gongbeak",
  "resonant gong-like bird call, metallic bell tones, single loud strike with a long decay, jungle",
  10
 ],
 [
  "Squeakmoth",
  "squeaky rubber-toy squeals of a small flying creature, playful, high pitched, short",
  2
 ],
 [
  "Rootwhale",
  "enormous slow underground whale-like drone, deep sub-bass moan, rumbling earth, eerie",
  15
 ],
 [
  "Glasscricket",
  "glassy tinkling cricket song, delicate chimes, rhythmic ticking, night jungle",
  7
 ],
 [
  "Bramblehoot",
  "questioning owl-like hoot with a wooden, hollow tone, repeated twice, forest night",
  9
 ],
 [
  "Sunsnort",
  "wet snorting and huffing of a large happy animal, warm breaths, rustling leaves",
  4
 ],
 [
  "Marshchoir",
  "chorus of many different frogs and insects singing together in a swamp at night, layered, dense",
  20
 ],
 [
  "Thornbark",
  "aggressive dog-like barking growl mixed with creaking branches, territorial, rainforest",
  6
 ],
 [
  "Lantern-sigher",
  "long breathy sigh and gentle whistling wind-like creature call, ethereal, soft, night",
  11
 ],
 [
  "Drip-Drummer",
  "rhythmic dripping and tapping of a creature drumming on hollow bamboo, rain drops, percussive",
  3
 ]
]

## Generate

- `negative_prompt` pushes away from low-quality / speech / music results.
- `num_inference_steps=100` is a compromise (the course notebook used 200).
- `audio_end_in_s` sets the duration. The model always generates 47 s internally and crops, so I also apply a short fade-out to hide the cut.
- I generate 3 candidates per creature and keep the **loudest-envelope-but-not-clipping** one automatically; listen to all three if you want to choose by ear (`KEEP_ALL = True`).

In [ ]:
NEGATIVE = "low quality, distorted, human speech, music, singing"
KEEP_ALL = False
os.makedirs("sounds", exist_ok=True)

def fade_out(wave, sr, secs=0.25):
    n = min(int(sr * secs), wave.shape[0])
    wave = wave.copy(); wave[-n:] *= np.linspace(1, 0, n)[:, None]
    return wave

for idx, (name, prompt, dur) in enumerate(VOICES):
    g = torch.Generator("cuda").manual_seed(1000 + idx)
    audios = pipe(prompt, negative_prompt=NEGATIVE, num_inference_steps=100,
                  audio_end_in_s=float(dur), num_waveforms_per_prompt=3, generator=g).audios
    waves = [a.T.float().cpu().numpy() for a in audios]
    # pick the candidate with the most energy (avoids near-silent results)
    best = max(range(3), key=lambda k: float(np.sqrt((waves[k] ** 2).mean())))
    for k, w in enumerate(waves):
        if not KEEP_ALL and k != best: continue
        w = w / max(np.abs(w).max(), 1e-8) * 0.95          # peak-normalise
        w = fade_out(w, SR)
        path = f"sounds/sound_{idx:02d}.wav" if not KEEP_ALL else f"sounds/sound_{idx:02d}_{k}.wav"
        sf.write(path, w, SR)
    print(f"{idx:02d} {name:<15} {dur:>2}s")
    display(Audio(f"sounds/sound_{idx:02d}.wav" if not KEEP_ALL else f"sounds/sound_{idx:02d}_{best}.wav"))

## Download
Put the `.wav` files in the `sounds/` folder of the repo.

In [ ]:
zip_path = shutil.make_archive("sounds", "zip", "sounds")
try:
    from google.colab import files; files.download(zip_path)
except ImportError:
    print("Saved", zip_path)

## Ideas to improve the sounds
- Layer two generations (e.g. a "frog" and a "whale") for stranger hybrids.
- Pitch-shift or time-stretch with `librosa` to make them feel more alien.
- Compare with Stable Audio 3, or condition on audio (audio-to-audio) from a real animal recording.